# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 2) Training Deep Neural Network

#### Let us rebuild the image classifier from the last chapter, but this time include regularization, learning rate schedulers, and faster optimizers. 

In [ ]:
import torch 
import torchvision
import torchvision.transforms.v2 as T 

from torch.utils.data import DataLoader

# Let us first insert the dataset preparation code from ImageClassifier.ipynb
torch.manual_seed(42)

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)
train_data, val_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)
test_loader = DataLoader(test_data, batch_size=32)

#### 1. Kaiming He Uniform Initialization + LayerNorm

In [ ]:
import torch.nn as nn
import torchmetrics 

from HelperMethods import HelperML 

# Linear layer initializes its weights using Kaiming uniform initialization and scales them down by (6)^0.5, which is not optimal, and which we can change using kaiming uniform (or similar) explicitly

import torch.nn as nn 

def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        #nn.init.zeros_(module.bias)


class ImageClassifier(nn.Module):

    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1, bias=False),
            #nn.BatchNorm1d(n_hidden1, momentum=bn_momentum),
            nn.LayerNorm(n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2, bias=False),
            #nn.BatchNorm1d(n_hidden2, momentum=bn_momentum),
            nn.LayerNorm(n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes, bias=False))
        
        self.mlp.apply(use_he_init)

    def forward(self, X):
        return self.mlp(X)
    
torch.manual_seed(42)

X_sample, y_sample = train_data[0]
n_inputs = X_sample.shape[1]*X_sample.shape[2]
print(f"n_inputs is {n_inputs}")
n_hidden1 = 100
n_hidden2 = 100
n_classes = 10
bn_momentum = 0.01
lr = 0.05

if torch.cuda.is_available():
    device = "cuda"
    
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device is {device}")

criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

helperML = HelperML(device)

#### Let's select the best optimizer 

In [ ]:
n_epochs = 5

##### i) SGD

In [ ]:
model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr)
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)

##### ii) SGD with Momentum and Nesterov Accelerated Gradient (NAG)

In [ ]:
model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)

##### iii) RMSprop optimizer

In [ ]:
model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.9)
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)

##### iv) Adam optimizer

In [ ]:
model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=lr)
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs)

#### 2. Kaiming He Initialization + LayerNorm + Cosine Annealing (for LR) with Warm Restarts

In [ ]:
# We saw that SGD Momentum optimizer performed the best
model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)

"""
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.1)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1., total_iters=3)
"""

cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=0.001)

n_epochs=20

# Adjust the train method in HelperMethods.HelperML class s.t. it uses scheduler
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs, cosine_repeat_scheduler)

#### 3. Regularize Model using Dropout

In [ ]:
# Modify class code of ImageClassifier s.t. it uses dropout

import torch.nn as nn
import torchmetrics 

from HelperMethods import HelperML 

import torch.nn as nn 

def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        #nn.init.zeros_(module.bias)


class ImageClassifier(nn.Module):

    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1, bias=False),
            #nn.BatchNorm1d(n_hidden1, momentum=bn_momentum),
            nn.LayerNorm(n_hidden1),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(n_hidden1, n_hidden2, bias=False),
            #nn.BatchNorm1d(n_hidden2, momentum=bn_momentum),
            nn.LayerNorm(n_hidden2),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(n_hidden2, n_classes, bias=False))
        
        self.mlp.apply(use_he_init)

    def forward(self, X):
        return self.mlp(X)
    
torch.manual_seed(42)

X_sample, y_sample = train_data[0]
n_inputs = X_sample.shape[1]*X_sample.shape[2]
print(f"n_inputs is {n_inputs}")
n_hidden1 = 100
n_hidden2 = 100
n_classes = 10
bn_momentum = 0.01
lr = 0.05

if torch.cuda.is_available():
    device = "cuda"
    
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device is {device}")

criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

helperML = HelperML(device)

model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes, bn_momentum)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=0.001)

n_epochs=20

helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs, cosine_repeat_scheduler)